# Ontology upload into Fuseki — how it works, tested end to end

Reference for getting RDF (`.ttl`, `.rdf`, `.owl`, …) into the Fuseki `ontology` dataset,
using the **`ontology_modeler`** component. Three parts:

1. **The upload model** — the Graph Store Protocol, content-type negotiation, the
   one-graph-per-file naming scheme, and why re-running never duplicates triples.
2. **A test suite** — self-contained checks that exercise the real classes.
3. **Asynchronous uploads** — `AsyncUploader` and a sync-vs-async benchmark.

Everything here is one component: `OntologyModeler` composes an `Uploader`, an
`AsyncUploader`, and the shared `FusekiClient` they run on. The RDF file helpers
(`CONTENT_TYPES`, `graph_uri`, `iter_rdf_files`) live in `ontology_modeler.rdf`.

**Prerequisites**
- Stack running: `cd infra/fuseki && docker compose up -d`
- Run from `src/Ontology Modeler/notebooks/`.

Test writes go into throwaway graphs under `urn:graph:__modeler_test__/` and are dropped;
your loaded FIBO data is untouched.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

MODULE_DIR = Path.cwd().parent            # src/Ontology Modeler (holds the package)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from ontology_modeler import OntologyModeler, FusekiClient, Uploader, AsyncUploader, UploadResult
from ontology_modeler.fuseki import FusekiError
from ontology_modeler.rdf import CONTENT_TYPES, graph_uri, iter_rdf_files
from ontology_modeler.config import REPO_ROOT

om = OntologyModeler()
client = om.fuseki
print("Fuseki up at", client.settings.base_url, "-> server time", client.ping())
print("dataset:", client.settings.dataset, "| repo root:", REPO_ROOT)

Fuseki up at http://localhost:3030 -> server time 2026-07-17T06:54:42.168+00:00
dataset: ontology | repo root: C:\Users\marci\OneDrive\DEV\EDU\AIML\Graph ML\Ontology Engineering


## 1. The upload model

### SPARQL Graph Store Protocol (GSP)

`FusekiClient` speaks GSP: a named graph is a resource addressed by URL and manipulated
with HTTP verbs. `Uploader` uses **`PUT`**, which *replaces* a graph — that is what makes
re-runs idempotent (`POST` would append and double the triples).

| Verb | Effect | Client method |
|------|--------|---------------|
| `PUT` | replace graph | `client.put_graph` / `put_file` |
| `GET` | read graph | `client.get_graph` |
| `DELETE` | drop graph | `client.delete_graph` |

### Content-type negotiation

Fuseki picks its parser from the `Content-Type`, not the extension. The `CONTENT_TYPES`
map is that translation, and `Uploader` sends it for you.

In [2]:
pd.DataFrame(sorted(CONTENT_TYPES.items()), columns=["extension", "Content-Type sent to Fuseki"])

,extension,Content-Type sent to Fuseki
0,.json,application/ld+json
1,.jsonld,application/ld+json
2,.n3,text/n3
3,.nq,application/n-quads
4,.nt,application/n-triples
5,.owl,application/rdf+xml
6,.rdf,application/rdf+xml
7,.trig,application/trig
8,.ttl,text/turtle
9,.turtle,text/turtle


### One named graph per file

Every file lands in its own graph, named from its **repo-relative path** — stable (so PUT
idempotency is meaningful) and readable (only `/` left unescaped). The `ontology` dataset
sets `unionDefaultGraph`, so an unqualified query sees the union of all files as one.

In [3]:
example = REPO_ROOT / "Ontology Repository/FIBO/fibo/FND/Utilities/AnnotationVocabulary.rdf"
print("graph URI:", graph_uri(example))
print("stable across calls:", graph_uri(example) == graph_uri(example))
print("spaces escaped:", "%20" in graph_uri(example) and " " not in graph_uri(example))

graph URI: urn:graph:Ontology%20Repository/FIBO/fibo/FND/Utilities/AnnotationVocabulary.rdf
stable across calls: True
spaces escaped: True


## 2. Anatomy of one upload

A single PUT into a throwaway graph, read straight back — all through the shared client.

In [4]:
TEST = "urn:graph:__modeler_test__/"

def put_ttl(graph, turtle):
    client.put_graph(graph, turtle.encode("utf-8"), content_type="text/turtle")

def count_graph(graph):
    rows = client.select(f"SELECT (COUNT(*) AS ?n) WHERE {{ GRAPH <{graph}> {{ ?s ?p ?o }} }}")
    return int(rows[0]["n"])

demo = TEST + "anatomy"
put_ttl(demo, """@prefix ex: <http://example.org/> .
ex:CurrentAccount a ex:DepositAccount ; ex:label "current account" .
ex:DepositAccount a ex:FinancialProduct .""")
print(f"after PUT: {count_graph(demo)} triples")
client.delete_graph(demo)
print(f"after DROP: {count_graph(demo)} triples")

after PUT: 3 triples


after DROP: 0 triples


## 3. Test suite

A tiny runner (no pytest). **Pure** tests pin the conventions (`CONTENT_TYPES`,
`graph_uri`, `iter_rdf_files`); **live** tests assert idempotency, replace semantics, and
error handling against Fuseki — all scoped to `urn:graph:__modeler_test__/`.

In [5]:
_TESTS = []
def test(fn): _TESTS.append(fn); return fn
def run_tests():
    rows, passed = [], 0
    for fn in _TESTS:
        name = fn.__name__.replace("test_", "")
        try:
            fn(); rows.append((name, "PASS", "")); passed += 1
        except AssertionError as e:
            rows.append((name, "FAIL", str(e) or "assertion failed"))
        except Exception as e:
            rows.append((name, "ERROR", f"{type(e).__name__}: {e}"))
    print(f"\n{passed}/{len(_TESTS)} passed")
    return pd.DataFrame(rows, columns=["test", "result", "detail"])

In [6]:
# --- Pure: the conventions everything depends on --------------------------

@test
def test_content_types_cover_common_rdf():
    for ext in (".ttl", ".rdf", ".owl", ".nt", ".jsonld", ".trig"):
        assert ext in CONTENT_TYPES, f"{ext} missing"
    assert CONTENT_TYPES[".ttl"] == "text/turtle"
    assert CONTENT_TYPES[".rdf"] == "application/rdf+xml"

@test
def test_graph_uri_is_stable_and_escaped():
    f = REPO_ROOT / "Ontology Repository/FIBO/fibo/FND/Utilities/AnnotationVocabulary.rdf"
    g = graph_uri(f)
    assert g == graph_uri(f) and g.startswith("urn:graph:")
    assert "%20" in g and " " not in g

@test
def test_graph_uri_is_repo_relative():
    g = graph_uri(REPO_ROOT / "Ontology Repository/FIBO/fibo/FND/AllFND.rdf")
    assert "C:" not in g and "Users" not in g, f"absolute path leaked: {g}"
    assert g.endswith("FND/AllFND.rdf")

@test
def test_iter_rdf_files_excludes_git_and_catalogs():
    files = iter_rdf_files([REPO_ROOT / "Ontology Repository/FIBO/fibo"])
    names = {f.name for f in files}
    assert files and "catalog-v001.xml" not in names
    assert not any(".git" in f.parts for f in files)
    assert all(f.suffix.lower() in CONTENT_TYPES for f in files)

@test
def test_iter_rdf_files_sorted_and_deduped():
    d = REPO_ROOT / "Ontology Repository/FIBO/fibo/FND"
    files = iter_rdf_files([d, d])
    assert files == sorted(set(files))

print("pure tests registered")

pure tests registered


In [7]:
# --- Live: real PUTs via the shared client, isolated to test graphs -------

@test
def test_roundtrip_upload_then_query():
    g = TEST + "roundtrip"
    put_ttl(g, "@prefix ex: <http://example.org/> .\nex:a ex:b ex:c .")
    try: assert count_graph(g) == 1
    finally: client.delete_graph(g)

@test
def test_put_is_idempotent():
    g = TEST + "idempotent"
    ttl = "@prefix ex: <http://example.org/> .\nex:a ex:b ex:c .\nex:a ex:b ex:d ."
    put_ttl(g, ttl); first = count_graph(g)
    put_ttl(g, ttl); second = count_graph(g)
    try: assert first == second == 2, f"{first} then {second}"
    finally: client.delete_graph(g)

@test
def test_put_replaces_not_appends():
    g = TEST + "replace"
    put_ttl(g, "@prefix ex: <http://example.org/> .\nex:a ex:b ex:c .")
    put_ttl(g, "@prefix ex: <http://example.org/> .\nex:x ex:y ex:z .")
    try: assert count_graph(g) == 1, "PUT should replace"
    finally: client.delete_graph(g)

@test
def test_malformed_rdf_is_rejected_and_isolated():
    g = TEST + "malformed"
    try:
        client.put_graph(g, b"@prefix ex: <http://example.org/> .\nex:a ex:b <<<bad>>>")
        assert False, "malformed RDF should raise FusekiError"
    except FusekiError as e:
        assert e.status >= 400
    assert count_graph(g) == 0, "a rejected upload must leave no triples"
    client.delete_graph(g)

@test
def test_uploader_uploads_a_real_file():
    up = om.uploader
    f = REPO_ROOT / "Ontology Repository/FIBO/fibo/FND/Utilities/AnnotationVocabulary.rdf"
    path, err = up.upload_file(f)
    assert err is None, f"upload failed: {err}"

print("live tests registered")

live tests registered


In [8]:
results = run_tests(); results


10/10 passed


,test,result,detail
0,content_types_cover_common_rdf,PASS,
1,graph_uri_is_stable_and_escaped,PASS,
2,graph_uri_is_repo_relative,PASS,
3,iter_rdf_files_excludes_git_and_catalogs,PASS,
4,iter_rdf_files_sorted_and_deduped,PASS,
5,roundtrip_upload_then_query,PASS,
6,put_is_idempotent,PASS,
7,put_replaces_not_appends,PASS,
8,malformed_rdf_is_rejected_and_isolated,PASS,
9,uploader_uploads_a_real_file,PASS,


## 4. Asynchronous uploads

`Uploader` does one PUT at a time. For hundreds of small files most of the wall-clock is
*waiting* (disk, TCP, parsing), which overlaps under `asyncio`. `AsyncUploader` uses
`aiohttp` with a semaphore bounding in-flight PUTs.

**Why bounded:** TDB2 serialises writers, so the server-side write is never parallel; a
handful of connections captures the overlap and more just contends on the write lock. Both
uploaders share the component's `FusekiClient` settings.

Notebooks run inside an event loop, so `nest_asyncio` lets us `await` in a cell.

In [9]:
import nest_asyncio; nest_asyncio.apply()

module = REPO_ROOT / "Ontology Repository/FIBO/fibo/FND"
# PUT is idempotent, so re-uploading replaces the same graphs — FIBO totals unchanged.
result: UploadResult = await om.async_uploader.upload_paths([module], concurrency=8)
print(f"async: {result}")
for f, err in result.failed:
    print("  FAIL", f.name, err)

async: uploaded 59/59 in 9.4s


## 5. Benchmark — sync vs async

Same files, same idempotent PUTs. Sync = `Uploader.upload_file` in a loop; async at a few
concurrency levels. Expect a win from 1→8, then flattening as the TDB2 write lock (not the
network) becomes the ceiling. On localhost with tiny files the gap is modest — the *shape*
is the point.

In [10]:
files = iter_rdf_files([module])
print(f"benchmarking {len(files)} files\n")

start = time.monotonic(); ok = 0
for f in files:
    _, err = om.uploader.upload_file(f); ok += err is None
sync_secs = time.monotonic() - start
print(f"sync (1 at a time):    {ok}/{len(files)} in {sync_secs:.2f}s")

rows = [("sync", 1, sync_secs)]
for c in (4, 8, 16):
    r = await om.async_uploader.upload_paths([module], concurrency=c)
    print(f"async (concurrency={c:2d}): {r.ok}/{r.total} in {r.elapsed:.2f}s")
    rows.append(("async", c, r.elapsed))

bench = pd.DataFrame(rows, columns=["mode", "concurrency", "seconds"])
bench["speedup_vs_sync"] = (sync_secs / bench["seconds"]).round(2)
bench

benchmarking 59 files



sync (1 at a time):    59/59 in 8.79s


async (concurrency= 4): 59/59 in 9.15s


async (concurrency= 8): 59/59 in 7.83s


async (concurrency=16): 59/59 in 7.82s


,mode,concurrency,seconds,speedup_vs_sync
0,sync,1,8.789543,1.00
1,async,4,9.152939,0.96
2,async,8,7.829105,1.12
3,async,16,7.822707,1.12


## Summary

- **Upload = HTTP PUT per file into `urn:graph:<repo-relative-path>`**, via `Uploader` /
  `AsyncUploader` over one shared `FusekiClient`. PUT gives idempotent re-runs.
- **Content-Type, not extension, drives parsing** — the `CONTENT_TYPES` map is that
  translation, pinned by the tests.
- **Async helps because uploads are wait-bound, up to the TDB2 write-lock ceiling.**

```python
from ontology_modeler import OntologyModeler
om = OntologyModeler()
om.uploader.upload_paths(["Ontology Repository/FIBO/fibo"], clear=True)   # sync
await om.async_uploader.upload_paths([...], concurrency=8)                # async
```
CLI: `python -m ontology_modeler upload --clear "Ontology Repository/FIBO/fibo"`
(add `--async` for the concurrent path).